## H03 验收运行事实

对应研究卷宗 README 的 H03 Acceptance；本 Notebook 保存固定样本的输入、执行和输出事实。
研究结论、状态与采用决定只由 README 的 H03 段落拥有。

2026-09-07：按第一性原理重设计后的候选版本，沿用既定四组成功样本及缺失 T+1 输入负例。
本次 runtime 清单增加共享 H02 schema 的定义文件；完整源码和输入仍随证据归档。
只有本次从头执行完成后的输出才证明当前版本的运行事实，历史结果见 README。


## Context & Methods

目标粒度是 `(symbol, trade_date, decision_ts_utc)`。所有 wall-clock 边界使用 `Asia/Shanghai`。输入来自只读正式 storage root；Notebook 将所需 payload 复制到新建的 `/tmp` 隔离 root，并重新提交无 upstream 的隔离 Meta。正式 payload 和 Meta 在运行前后做 SHA-256、size 与 mtime 复核。

### Key Assumptions

- 固定 V1 输入版本为两市 `stock_trade_1m/v1` 与 `adj_factor/v1`。
- 有效 H03 Meta 直接复用；不根据当前输入重新证明可复用。
- 预注册最终样本为 `2025-12-31 -> 2026-01-05`、`2026-04-30 -> 2026-05-06`、`2026-07-27 -> 2026-07-28`；`2025-11-18 -> 2025-11-19` 是 smoke，`2025-11-24 -> 2025-11-25` 是已知缺失输入的负例。

### 本轮执行与恢复

- 在仓库根目录运行，kernel 必须使用项目 `.venv/bin/python`，由 `uv.lock` 锁定。
- 显式设置 `H03_EVIDENCE_ROOT` 为已存在的本地证据目录；每次执行创建独立子目录。
- 全部仓库受版本控制的文件及未忽略候选文件复制成代码快照；CLI 在该快照中执行。
  快照使用固定无效凭证 `.env.dev`，子进程只接收明确列出的环境变量，不携带业务凭证。
- 当前验证仍沿用上述预注册日期，不重新选择样本。没有模型选择或随机计算；固定
  `PYTHONHASHSEED=0`。代码、输入、命令和全部失败 stderr 随执行记录保存。
- 本轮不设置资源 SLA；wall time 和 peak RSS 只作为运行事实。


In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from src.access import Access, meta
from src.utils.path import ObjectPaths, PathManager

repo_root = Path.cwd().resolve()
assert (repo_root / 'pyproject.toml').is_file(), 'Run the kernel in the repository root'
evidence_root = Path(os.environ['H03_EVIDENCE_ROOT']).resolve(strict=True)
run_root = Path(tempfile.mkdtemp(prefix='run-', dir=evidence_root))
validation_root = run_root / 'store'
validation_root.mkdir()
candidate_root = run_root / 'candidate'
candidate_root.mkdir()
source_names = subprocess.check_output(
    ['git', 'ls-files', '--cached', '--others', '--exclude-standard'],
    cwd=repo_root, text=True,
).splitlines()
source_names = sorted({name for name in source_names if (repo_root / name).is_file()})
for name in source_names:
    source_path = repo_root / name
    if source_path.is_file():
        target_path = candidate_root / name
        target_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_path, target_path)
(candidate_root / '.env.dev').write_text(
    'FTP_HOST=example.invalid\nFTP_USER=validation\n'
    'FTP_PASSWORD=unused\nTUSHARE_TOKEN=unused\n', encoding='utf-8',
)
runtime_names = (
    'src/access/access.py', 'src/cli.py',
    'src/data_system/builders/stock_1430.py',
    'src/data_system/builders/level2_stock_trade_1m.py',
    'src/data_system/steps/_partition.py',
    'src/data_system/steps/stock_1430_build.py',
    'src/jobs/requests.py', 'src/workflows/offline_daily_data.py',
)
runtime_manifest = ''.join(
    f'{hashlib.sha256((candidate_root / name).read_bytes()).hexdigest()}  {name}\n'
    for name in runtime_names
)
runtime_sha256 = hashlib.sha256(runtime_manifest.encode()).hexdigest()
source_manifest = {
    name: hashlib.sha256((candidate_root / name).read_bytes()).hexdigest()
    for name in sorted(set(source_names))
    if name.startswith(('src/', 'tests/')) or name in ('pyproject.toml', 'uv.lock')
}
(run_root / 'code-manifest.json').write_text(
    json.dumps(source_manifest, indent=2, sort_keys=True), encoding='utf-8'
)
formal_root = Path('/home/wsw/app/data')
assert all((formal_root / name).is_dir() for name in (
    'raw', 'staging', 'processed', 'features', 'labels', 'experiments'
))
formal_pm = PathManager(formal_root)
validation_pm = PathManager(validation_root)
case_pairs = {
    'smoke': ('2025-11-18', '2025-11-19'),
    'negative_missing_t1': ('2025-11-24', '2025-11-25'),
    'final_year_boundary': ('2025-12-31', '2026-01-05'),
    'final_holiday_boundary': ('2026-04-30', '2026-05-06'),
    'final_regular_pair': ('2026-07-27', '2026-07-28'),
}
environment_record = {
    'python': sys.version, 'executable': sys.executable,
    'numpy': np.__version__, 'pandas': pd.__version__, 'pyarrow': pa.__version__,
    'head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'runtime_sha256': runtime_sha256,
    'uv_lock_sha256': source_manifest['uv.lock'],
    'case_pairs': case_pairs,
    'run_root': str(run_root), 'formal_root': str(formal_root),
}
(run_root / 'environment.json').write_text(
    json.dumps(environment_record, indent=2), encoding='utf-8'
)
print(environment_record)


{'python': '3.13.13 (main, Apr 14 2026, 14:28:56) [Clang 22.1.3 ]', 'executable': '/home/wsw/app/dev/trading/.venv/bin/python', 'numpy': '2.5.1', 'pandas': '3.0.2', 'pyarrow': '25.0.0', 'head': '57d94ea073d1736e9d40e1126933f37978d6be48', 'runtime_sha256': '762624ec042906b7c6bcde36baf40c4bde4cd51a4bbc2e830b438dd42d0ea4bb', 'uv_lock_sha256': '96125da32034e999352f8a0326f6bddb0c5a9408c8880a9197f2faebf5d4f512', 'case_pairs': {'smoke': ('2025-11-18', '2025-11-19'), 'negative_missing_t1': ('2025-11-24', '2025-11-25'), 'final_year_boundary': ('2025-12-31', '2026-01-05'), 'final_holiday_boundary': ('2026-04-30', '2026-05-06'), 'final_regular_pair': ('2026-07-27', '2026-07-28')}, 'run_root': '/tmp/stock-1430-h03-redesign-cb53htly/run-cxq87ato', 'formal_root': '/home/wsw/app/data'}


## Data

### 1. Copy bounded formal inputs into the isolated root

In [2]:
def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _copy_formal_object(source_paths: ObjectPaths, target_paths: ObjectPaths, *, role: str, partition: str) -> dict[str, object]:
    record = meta.require(
        pm=formal_pm,
        meta_path=source_paths.meta_path,
        expected_payload_path=source_paths.payload_path,
    )
    target_paths.payload_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(record.payload_path, target_paths.payload_path)
    meta.commit(pm=validation_pm, payload_path=target_paths.payload_path)
    return {
        'role': role,
        'partition': partition,
        'formal_meta': source_paths.meta_path.relative_to(formal_root).as_posix(),
        'meta_sha256': _sha256_file(source_paths.meta_path),
        'payload_sha256': _sha256_file(record.payload_path),
        'payload_bytes': record.size_bytes,
        'meta_path': source_paths.meta_path,
        'payload_path': record.payload_path,
    }


input_rows = []
for calendar_year in (2025, 2026):
    input_rows.append(
        _copy_formal_object(
            formal_pm.processed_year_object(
                dataset_name='trade_calendar', version='v1', calendar_year=calendar_year
            ),
            validation_pm.processed_year_object(
                dataset_name='trade_calendar', version='v1', calendar_year=calendar_year
            ),
            role='calendar',
            partition=str(calendar_year),
        )
    )

all_dates = sorted({value for pair in case_pairs.values() for value in pair})
missing_inputs = []
for trade_date in all_dates:
    for dataset_name in ('sh_stock_trade_1m', 'sz_stock_trade_1m', 'adj_factor'):
        source_paths = formal_pm.processed_object(
            dataset_name=dataset_name, version='v1', trade_date=trade_date
        )
        target_paths = validation_pm.processed_object(
            dataset_name=dataset_name, version='v1', trade_date=trade_date
        )
        try:
            input_rows.append(
                _copy_formal_object(
                    source_paths,
                    target_paths,
                    role=dataset_name,
                    partition=trade_date,
                )
            )
        except FileNotFoundError:
            missing_inputs.append((dataset_name, trade_date))

assert sorted(missing_inputs) == [
    ('sh_stock_trade_1m', '2025-11-25'),
    ('sz_stock_trade_1m', '2025-11-25'),
]
formal_snapshots = {
    path: (_sha256_file(path), path.stat().st_size, path.stat().st_mtime_ns)
    for row in input_rows
    for path in (row['meta_path'], row['payload_path'])
}
input_identity = pd.DataFrame(input_rows).drop(columns=['meta_path', 'payload_path'])
manifest_text = input_identity.sort_values(['role', 'partition']).to_json(orient='records')
input_manifest_sha256 = hashlib.sha256(manifest_text.encode('utf-8')).hexdigest()
print({'copied_objects': len(input_identity), 'missing_inputs': missing_inputs, 'input_manifest_sha256': input_manifest_sha256})
input_identity.head(12)
(run_root / 'input-manifest.json').write_text(manifest_text, encoding='utf-8')


{'copied_objects': 30, 'missing_inputs': [('sh_stock_trade_1m', '2025-11-25'), ('sz_stock_trade_1m', '2025-11-25')], 'input_manifest_sha256': 'cfe55cf061bb6b42cecf8c6e0520d9680ec90aab0ece38b1b6405a7dd14b54e7'}


9573

## Results

### 2. Run smoke, known-negative, and preregistered final cases

In [3]:
def _run_backfill(case_name: str, trade_date: str) -> dict[str, object]:
    environment = {
        'PATH': os.environ['PATH'], 'HOME': os.environ['HOME'],
        'ENV': 'dev', 'ZERO_STORAGE_ROOT': str(validation_root),
        'PYTHONHASHSEED': '0', 'PYTHONDONTWRITEBYTECODE': '1',
    }
    command = [
        '/usr/bin/time',
        '-v',
        sys.executable,
        '-m',
        'src.cli',
        'data-stock-1430-backfill',
        '--start',
        trade_date,
        '--end',
        trade_date,
    ]
    started = perf_counter()
    completed = subprocess.run(
        command,
        cwd=candidate_root,
        env=environment,
        text=True,
        capture_output=True,
        check=False,
    )
    elapsed_seconds = perf_counter() - started
    rss_match = re.search(
        r'Maximum resident set size \(kbytes\): (\d+)', completed.stderr
    )
    return {
        'command': command, 'cwd': str(candidate_root), 'environment': environment,
        'case': case_name,
        'trade_date': trade_date,
        'returncode': completed.returncode,
        'wall_seconds': round(elapsed_seconds, 3),
        'peak_rss_kib': int(rss_match.group(1)) if rss_match else None,
        'stderr': completed.stderr,
    }


primary_runs = [
    _run_backfill(case_name, pair[0])
    for case_name, pair in case_pairs.items()
]
for result in primary_runs:
    if result['case'] == 'negative_missing_t1':
        assert result['returncode'] != 0
        assert '2025-11-25' in result['stderr']
        assert 'required Meta' in result['stderr']
    else:
        assert result['returncode'] == 0, result['stderr'][-2000:]
negative_feature = validation_pm.feature_object(
    feature_set='l2_stock_1430', version='v1', trade_date='2025-11-24'
)
negative_label = validation_pm.label_object(
    label_set='l2_stock_1430_t1_vwap_rank', version='v1', trade_date='2025-11-24'
)
assert negative_feature.meta_path.is_file()
assert not negative_label.meta_path.exists()
display(pd.DataFrame(primary_runs).loc[:, ['case', 'trade_date', 'returncode', 'wall_seconds', 'peak_rss_kib']])
(run_root / 'primary-runs.json').write_text(
    json.dumps(primary_runs, indent=2), encoding='utf-8'
)


,case,trade_date,returncode,wall_seconds,peak_rss_kib
0,smoke,2025-11-18,0,2.725,1097312
1,negative_missing_t1,2025-11-24,1,2.355,869016
2,final_year_boundary,2025-12-31,0,2.704,1074608
3,final_holiday_boundary,2026-04-30,0,2.730,1128264
4,final_regular_pair,2026-07-27,0,2.667,1087764


28489

### 3. Validate schema, keys, ranges, coverage, nulls, and output identities

In [4]:
key_columns = ('symbol', 'trade_date', 'decision_ts_utc')
windows = (5, 15, 30, 60)
metric_stems = (
    'edge_vwap_return_rank',
    'high_low_range_rank',
    'notional_rank',
    'trade_count_rank',
    'average_trade_notional_rank',
    'tick_signed_volume_ratio_rank',
    'tick_signed_notional_ratio_rank',
)
feature_columns = tuple(
    column
    for window in windows
    for column in (
        *(f'f_l2_{stem}_{window}m' for stem in metric_stems),
        f'f_l2_observed_minute_ratio_{window}m',
    )
)
observed_columns = tuple(f'f_l2_observed_minute_ratio_{window}m' for window in windows)
rank_columns = tuple(column for column in feature_columns if column not in observed_columns)
expected_feature_schema = pa.schema([
    pa.field('symbol', pa.string(), nullable=False),
    pa.field('trade_date', pa.string(), nullable=False),
    pa.field('decision_ts_utc', pa.int64(), nullable=False),
    *[
        pa.field(column, pa.float64(), nullable=column not in observed_columns)
        for column in feature_columns
    ],
])
expected_label_schema = pa.schema([
    pa.field('symbol', pa.string(), nullable=False),
    pa.field('trade_date', pa.string(), nullable=False),
    pa.field('decision_ts_utc', pa.int64(), nullable=False),
    pa.field('y_rank_return', pa.float64(), nullable=True),
])


def _key_digest(frame: pd.DataFrame) -> str:
    encoded = ''.join(
        f'{symbol}|{trade_date}|{decision_ts_utc}\n'
        for symbol, trade_date, decision_ts_utc in frame.loc[:, key_columns].itertuples(
            index=False, name=None
        )
    ).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


profile_rows = []
sparsity_rows = []
successful_cases = {name: pair for name, pair in case_pairs.items() if name != 'negative_missing_t1'}
for case_name, (trade_date, next_trade_date) in successful_cases.items():
    feature_paths = validation_pm.feature_object(
        feature_set='l2_stock_1430', version='v1', trade_date=trade_date
    )
    label_paths = validation_pm.label_object(
        label_set='l2_stock_1430_t1_vwap_rank', version='v1', trade_date=trade_date
    )
    feature_meta = meta.require(
        pm=validation_pm, meta_path=feature_paths.meta_path,
        expected_payload_path=feature_paths.payload_path,
    )
    label_meta = meta.require(
        pm=validation_pm, meta_path=label_paths.meta_path,
        expected_payload_path=label_paths.payload_path,
    )
    assert feature_meta.upstream is None and feature_meta.symbol_slices is None
    assert label_meta.upstream is None and label_meta.symbol_slices is None
    with pq.ParquetFile(feature_meta.payload_path) as parquet_file:
        feature = parquet_file.read()
    with pq.ParquetFile(label_meta.payload_path) as parquet_file:
        label = parquet_file.read()
    assert feature.schema.equals(expected_feature_schema, check_metadata=False)
    assert label.schema.equals(expected_label_schema, check_metadata=False)
    access = Access(pm=validation_pm, processed_version='v1')
    assert access.next_trade_date(trade_date=trade_date) == next_trade_date
    expected_decision_ts_utc = int(
        pd.Timestamp(f'{trade_date} 14:30', tz='Asia/Shanghai').timestamp() * 1_000_000
    )
    assert feature.column('decision_ts_utc').to_pylist() == [
        expected_decision_ts_utc
    ] * feature.num_rows
    minute_keys = access.stock_trade_minutes(trade_date=trade_date).select([
        'symbol', 'minute_start_ts_utc', 'phase'
    ]).to_pandas()
    expected_symbols = sorted(minute_keys.loc[
        minute_keys['phase'].eq(2)
        & minute_keys['minute_start_ts_utc'].lt(expected_decision_ts_utc),
        'symbol',
    ].unique())
    assert feature.column('symbol').to_pylist() == expected_symbols
    feature_frame = feature.to_pandas()
    label_frame = label.to_pandas()
    assert len(feature_frame) > 0
    assert not feature_frame.loc[:, key_columns].isna().any().any()
    assert not feature_frame.duplicated(list(key_columns)).any()
    assert list(feature_frame.loc[:, key_columns].itertuples(index=False, name=None)) == sorted(
        feature_frame.loc[:, key_columns].itertuples(index=False, name=None)
    )
    assert label.select(list(key_columns)).equals(feature.select(list(key_columns)))
    for column in rank_columns:
        values = feature_frame[column].dropna().to_numpy(dtype=float)
        assert np.isfinite(values).all() and (values > 0).all() and (values <= 1).all()
        sparsity_rows.append({
            'case': case_name, 'trade_date': trade_date, 'column': column,
            'null_rate': float(feature_frame[column].isna().mean()),
        })
    for column in observed_columns:
        values = feature_frame[column].to_numpy(dtype=float)
        assert np.isfinite(values).all() and (values >= 0).all() and (values <= 1).all()
    label_values = label_frame['y_rank_return'].dropna().to_numpy(dtype=float)
    assert len(label_values) > 0
    assert np.isfinite(label_values).all() and (label_values > 0).all() and (label_values <= 1).all()
    row = {
        'case': case_name,
        'trade_date': trade_date,
        'next_trade_date': next_trade_date,
        'rows': len(feature_frame),
        'valid_labels': len(label_values),
        'label_null_rate': float(label_frame['y_rank_return'].isna().mean()),
        'key_sha256': _key_digest(feature_frame),
        'feature_payload_sha256': _sha256_file(feature_meta.payload_path),
        'label_payload_sha256': _sha256_file(label_meta.payload_path),
    }
    for window in windows:
        observed = feature_frame[f'f_l2_observed_minute_ratio_{window}m']
        row[f'observed_{window}m_mean'] = float(observed.mean())
        row[f'observed_{window}m_zero_rate'] = float(observed.eq(0).mean())
        row[f'observed_{window}m_full_rate'] = float(observed.eq(1).mean())
    profile_rows.append(row)

profile = pd.DataFrame(profile_rows)
sparsity = pd.DataFrame(sparsity_rows).sort_values(
    ['null_rate', 'trade_date', 'column'], ascending=[False, True, True]
)
display(profile)
print(profile.loc[:, ['case', 'key_sha256', 'feature_payload_sha256', 'label_payload_sha256']].to_dict('records'))
display(sparsity.head(16))
profile.to_json(run_root / 'profiles.json', orient='records', indent=2)
sparsity.to_json(run_root / 'sparsity.json', orient='records', indent=2)


,case,trade_date,next_trade_date,rows,valid_labels,label_null_rate,key_sha256,feature_payload_sha256,label_payload_sha256,observed_5m_mean,...,observed_5m_full_rate,observed_15m_mean,observed_15m_zero_rate,observed_15m_full_rate,observed_30m_mean,observed_30m_zero_rate,observed_30m_full_rate,observed_60m_mean,observed_60m_zero_rate,observed_60m_full_rate
0,smoke,2025-11-18,2025-11-19,5157,5149,0.001551,3dc21f7b20b78dcac96c8c2aa099010ba7d81757d3e4fd...,823ad241a3418417307595184268273f5e1c95eb3950e8...,ff90da3df6c1b5010683e33bb52ea0e00cc060d92aa5b1...,0.987318,...,0.959860,0.986336,0.000000,0.905177,0.985405,0.000000,0.854567,0.982567,0.0,0.770409
1,final_year_boundary,2025-12-31,2026-01-05,5170,5158,0.002321,ac408d882baf848e83137e37295e4509740d86ba1e69e8...,b2c872d166c72543be03ead155494265ab5a4e7167725e...,b5c97f9216947d1a6994bd5647af261d1ef22b57b880ba...,0.983366,...,0.942553,0.979123,0.000000,0.861315,0.979497,0.000000,0.810251,0.976593,0.0,0.721470
2,final_holiday_boundary,2026-04-30,2026-05-06,5150,5136,0.002718,f43c2e1e5ab369f0ce71ec04ce1cd97c8ed4fb3199ddff...,d602179df1074270443be33eb45cfee06ce1b83adb4365...,51d90885d14e913becee16c037aa80a5f8c8379ecb272d...,0.988000,...,0.965631,0.986511,0.000388,0.927184,0.985223,0.000194,0.881942,0.983997,0.0,0.818641
3,final_regular_pair,2026-07-27,2026-07-28,5192,5188,0.000770,abd4432917ecbb68f9f4d509c607fa3d475ec62647a3be...,62254906e509b25e176c8a8ccab7b89e124e8f2f6ef8e1...,1cad927f7ebc4ffc0d23cf59ea45ccd54ffdfec9f0af7d...,0.982435,...,0.939908,0.980508,0.000193,0.882512,0.974859,0.000000,0.784476,0.968740,0.0,0.678544


[{'case': 'smoke', 'key_sha256': '3dc21f7b20b78dcac96c8c2aa099010ba7d81757d3e4fd61b58f6a087601a777', 'feature_payload_sha256': '823ad241a3418417307595184268273f5e1c95eb3950e84243466ab1d331d846', 'label_payload_sha256': 'ff90da3df6c1b5010683e33bb52ea0e00cc060d92aa5b1c1545e43d403b94cc1'}, {'case': 'final_year_boundary', 'key_sha256': 'ac408d882baf848e83137e37295e4509740d86ba1e69e8d5a8370bed0e874f5d', 'feature_payload_sha256': 'b2c872d166c72543be03ead155494265ab5a4e7167725e23d107b37543f14e1d', 'label_payload_sha256': 'b5c97f9216947d1a6994bd5647af261d1ef22b57b880ba5584e96c81992ee9f1'}, {'case': 'final_holiday_boundary', 'key_sha256': 'f43c2e1e5ab369f0ce71ec04ce1cd97c8ed4fb3199ddff806850a4591134b849', 'feature_payload_sha256': 'd602179df1074270443be33eb45cfee06ce1b83adb43655580b59ea3848caaf4', 'label_payload_sha256': '51d90885d14e913becee16c037aa80a5f8c8379ecb272d80a82f0ec6d710bff3'}, {'case': 'final_regular_pair', 'key_sha256': 'abd4432917ecbb68f9f4d509c607fa3d475ec62647a3be27de076032e0d59

,case,trade_date,column,null_rate
105,final_regular_pair,2026-07-27,f_l2_edge_vwap_return_rank_60m,0.045069
98,final_regular_pair,2026-07-27,f_l2_edge_vwap_return_rank_30m,0.031202
84,final_regular_pair,2026-07-27,f_l2_edge_vwap_return_rank_5m,0.030431
35,final_year_boundary,2025-12-31,f_l2_edge_vwap_return_rank_15m,0.029207
91,final_regular_pair,2026-07-27,f_l2_edge_vwap_return_rank_15m,0.028891
28,final_year_boundary,2025-12-31,f_l2_edge_vwap_return_rank_5m,0.026886
49,final_year_boundary,2025-12-31,f_l2_edge_vwap_return_rank_60m,0.025145
21,smoke,2025-11-18,f_l2_edge_vwap_return_rank_60m,0.023463
0,smoke,2025-11-18,f_l2_edge_vwap_return_rank_5m,0.023075
42,final_year_boundary,2025-12-31,f_l2_edge_vwap_return_rank_30m,0.021277


### 4. Verify Meta-hit immutability and formal-input read-only behavior

In [5]:
def _object_snapshot(paths: ObjectPaths) -> dict[str, tuple[str, int, int]]:
    return {
        path.name + ':' + path.parent.as_posix(): (
            _sha256_file(path), path.stat().st_size, path.stat().st_mtime_ns
        )
        for path in (paths.payload_path, paths.meta_path)
    }


hit_runs = []
for case_name, (trade_date, _) in successful_cases.items():
    feature_paths = validation_pm.feature_object(
        feature_set='l2_stock_1430', version='v1', trade_date=trade_date
    )
    label_paths = validation_pm.label_object(
        label_set='l2_stock_1430_t1_vwap_rank', version='v1', trade_date=trade_date
    )
    before = {**_object_snapshot(feature_paths), **_object_snapshot(label_paths)}
    hit = _run_backfill(case_name + '_meta_hit', trade_date)
    assert hit['returncode'] == 0, hit['stderr'][-2000:]
    after = {**_object_snapshot(feature_paths), **_object_snapshot(label_paths)}
    assert after == before
    hit_runs.append(hit)

for path, before in formal_snapshots.items():
    assert (_sha256_file(path), path.stat().st_size, path.stat().st_mtime_ns) == before

hit_results = pd.DataFrame(hit_runs).loc[:, ['case', 'trade_date', 'returncode', 'wall_seconds', 'peak_rss_kib']]
display(hit_results)
print({'formal_files_unchanged': len(formal_snapshots), 'validation_root': str(validation_root)})
(run_root / 'meta-hit-runs.json').write_text(
    json.dumps(hit_runs, indent=2), encoding='utf-8'
)
for name, expected_sha256 in source_manifest.items():
    assert _sha256_file(repo_root / name) == expected_sha256
    assert _sha256_file(candidate_root / name) == expected_sha256
print({'source_files_unchanged': len(source_manifest), 'runtime_sha256': runtime_sha256})


,case,trade_date,returncode,wall_seconds,peak_rss_kib
0,smoke_meta_hit,2025-11-18,0,1.105,244096
1,final_year_boundary_meta_hit,2025-12-31,0,1.094,244492
2,final_holiday_boundary_meta_hit,2026-04-30,0,1.061,244516
3,final_regular_pair_meta_hit,2026-07-27,0,1.102,244444


{'formal_files_unchanged': 60, 'validation_root': '/tmp/stock-1430-h03-redesign-cb53htly/run-cxq87ato/store'}
{'source_files_unchanged': 226, 'runtime_sha256': '762624ec042906b7c6bcde36baf40c4bde4cd51a4bbc2e830b438dd42d0ea4bb'}


## 事实与边界

本次各样本的行数、有效监督值、null coverage、key/payload 摘要及资源用量见已执行输出。
输入 manifest、代码快照、环境和包含负例 stderr 的完整命令记录保存在输出所示的 run_root。
本 Notebook 只验证 H03 固定输入与样本的数据构建行为；不选择因子，不评价 alpha、收益或容量。
采用评审、研究状态和本轮验收结论见 README 的 H03 段落。


完整归档：`/home/wsw/app/research-evidence/stock-1430-h03-2026-09-05-8bc8bkvg.tar.gz`。其中保留源代码、固定输入和 CLI 命令记录，归档内的 `REPRODUCE.md` 提供恢复步骤。